## Comparison of early fusion (best model from Phase 1) and GNNs
Replace Morgan fingerprint encoder with GCN/GIN/GAT encoder\
Input: molecular graphs (instead of fingerprints)\
Output: 128-dim drug embedding (match fingerprint dimensionality)\
Keep everything downstream unchanged (fusion layer, genomics encoder, final prediction) 

In [7]:
import pandas as pd
import numpy as np
import torch
import warnings
import gc
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, GroupShuffleSplit, cross_val_predict
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
import sys
from pathlib import Path
from torch_geometric.loader import DataLoader

sys.path.append(str(Path.cwd().parent))
from configs.config import HARMONIZED_DATA_PATH
from gpu.gnn_models_utils import (
    smiles_to_graph,
    GCNDrugEncoder,
    GATDrugEncoder,
    GINDrugEncoder
)
molclr_path = os.path.abspath('../MolCLR')
if molclr_path not in sys.path:
    sys.path.append(molclr_path)
from models.gcn_molclr import GCN

In [1]:
def early_fusion(df, target, model_name, drug_feature_type='morgan', apply_pca=True):
    """
    target: 'AUC' or 'IC50'
    model_name: 'RandomForest' or 'ElasticNet'
    drug_feature_type: 'morgan' (Phase 1 Baseline) or 'gnn' (Phase 2 GNN-Embeddings)
    apply_pca: True -> compresses the active ingredient features into 10 PCs (identical to Phase 1)
               False -> uses all GNN dimensions directly (scaled only)
    """
    df = df.copy()
    
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 
                      'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist(), index=df.index).astype('float32')
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)

    if drug_feature_type == 'morgan':
        if 'Bit_0' not in df.columns:
            fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index).astype('uint8')
            fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
            df = pd.concat([df, fp_df], axis=1)
        drug_cols = [col for col in df.columns if col.startswith('Bit_')]
    elif drug_feature_type == 'gnn':
        drug_cols = [col for col in df.columns if col.startswith('GNN_')]
        assert len(drug_cols) > 0, "No 'GNN_*' columns found in df! Run add_gnn_embeddings_to_df() first."
    else:
        raise ValueError("drug_feature_type must be 'morgan' or 'gnn'.")

    gene_cols = list(df.filter(regex=r'.* \(.*\)').columns)

    X = pd.concat([df[pharmacophores], df[drug_cols], df[gene_cols]], axis=1)
    y = df[target]

    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df['DRUG_ID'].values))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups = df.iloc[train_idx]['DRUG_ID'].values

    # Preprocessing
    if apply_pca:
        drug_transformer = (
            Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=10, random_state=42))])
            if drug_feature_type == 'gnn'
            else PCA(n_components=10, random_state=42)
        )
    else:
        drug_transformer = StandardScaler()

    preprocessor = ColumnTransformer(
        transformers=[
            ('pharmacophore', StandardScaler(), pharmacophores),
            ('drug', drug_transformer, drug_cols)
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )
    preprocessor.set_output(transform="pandas")

    X_train_transformed = preprocessor.fit_transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    # 6. Downstream-Modelle (unverändert)
    if model_name == 'RandomForest':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features='sqrt',
            n_jobs=4,
            random_state=42
        )

        print(f"Starting Cross-Validation on Training Data ({drug_feature_type.upper()})...")
        cv_results = cross_validate(
            model, X_train_transformed, y_train,
            groups=groups,
            cv=GroupKFold(5),
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        rmse_scores = np.sqrt(-cv_results['test_neg_mean_squared_error'])
        r2_scores = cv_results['test_r2']

        print(f"--- Multimodal (Random Forest - Early Fusion | {drug_feature_type.upper()}) ---")
        print(f"CV R² Score:   {np.mean(r2_scores):.4f}")
        print(f"CV RMSE:       {np.mean(rmse_scores):.4f}")

        model.fit(X_train_transformed, y_train)
        y_pred = model.predict(X_test_transformed)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Test R² Score: {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        print("-" * 45)

        # Bugfix: X_train_transformed.columns statt X.columns verwenden
        all_features = X_train_transformed.columns
        importances = model.feature_importances_
        gene_set = set(gene_cols)

        features_df = pd.DataFrame({
            'Feature_Name': all_features,
            'Importance': importances,
            'Type': ['Genetic' if f in gene_set else 'Molecular structure' for f in all_features]
        })

        selected_features = features_df[features_df['Importance'] > 0.0001]
        print(f"Selected features: {len(selected_features)} / {len(all_features)} "
              f"(Chem: {(selected_features['Type'] == 'Molecular structure').sum()} | "
              f"Bio: {(selected_features['Type'] == 'Genetic').sum()})")
        print("\nTop 10 most important features:")
        print(features_df.sort_values(by='Importance', ascending=False).head(10))

        return {'cv_r2': np.mean(r2_scores), 'cv_rmse': np.mean(rmse_scores),
                'test_r2': test_r2, 'test_rmse': test_rmse}

    elif model_name == 'ElasticNet':
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1],
                cv=list(GroupKFold(n_splits=5).split(X_train_transformed, y_train, groups=groups)),
                max_iter=8000,
                alphas=20,
                tol=1e-3,
                random_state=42,
                n_jobs=1
            ))
        ])

        pipeline.fit(X_train_transformed, y_train)
        fitted_model = pipeline.named_steps['model']

        y_pred = pipeline.predict(X_test_transformed)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        # CV R² direkt aus dem kleinsten mittleren MSE über die Folds berechnen
        best_mean_mse = np.min(np.mean(fitted_model.mse_path_, axis=-1))
        train_cv_r2 = 1 - (best_mean_mse / np.var(y_train))

        print(f"\n--- Elastic Net (Early Fusion | {drug_feature_type.upper()}) ---")
        print(f"Chosen Alpha:    {fitted_model.alpha_:.6f} | L1-Ratio: {fitted_model.l1_ratio_:.2f}")
        print(f"CV R² Score:     {train_cv_r2:.4f}")
        print(f"Test R² Score:   {test_r2:.4f}")
        print(f"Test RMSE:       {test_rmse:.4f}")
        print("-" * 45)

        feature_names = X_train_transformed.columns
        features_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': fitted_model.coef_})
        selected_features = features_df[features_df['Coefficient'] != 0]

        print(f"Elastic Net selected {len(selected_features)} / {len(feature_names)} features.")
        print(f"Top 5 Positive:\n{features_df.sort_values(by='Coefficient', ascending=False).head(5)}")
        print(f"Top 5 Negative:\n{features_df.sort_values(by='Coefficient', ascending=True).head(5)}")

        return {'cv_r2': train_cv_r2, 'test_r2': test_r2, 'test_rmse': test_rmse}

In [6]:
@torch.no_grad()
def add_gnn_embeddings_to_df(df, model, device, batch_size=64):
    """
    Calculates the graph embedding once for each unique SMILES
    and appends the columns 'GNN_0' ... 'GNN_d' to a copy of df.
    """
    model = model.to(device).eval()
    
    unique_smiles = df['SMILES'].dropna().unique()
    graphs = []
    valid_smiles = []
    failed_smiles = []
    
    for s in unique_smiles:
        g = smiles_to_graph(s)
        if g is not None:
            graphs.append(g)
            valid_smiles.append(s)
        else:
            failed_smiles.append(s)
            
    if failed_smiles:
        print(f"Warning: {len(failed_smiles)} SMILES could not be converted to graphs.")
    
    # In Batches through the GNN
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    all_embs = []
    
    for batch in loader:
        batch = batch.to(device)
        _, h_graph = model(batch)
        all_embs.append(h_graph.cpu().numpy())
        
    emb_matrix = np.vstack(all_embs).astype(np.float32)
    emb_cols = [f'GNN_{i}' for i in range(emb_matrix.shape[1])]
    
    # Map back to the complete DataFrame using SMILES
    emb_lookup = pd.DataFrame(emb_matrix, columns=emb_cols)
    emb_lookup['SMILES'] = valid_smiles
    
    df_clean = df.drop(columns=[c for c in df.columns if c.startswith('GNN_')], errors='ignore')
    df_merged = df_clean.merge(emb_lookup, on='SMILES', how='inner')
    
    print(f"Embeddings created: {emb_matrix.shape[1]} dimensions for {len(valid_smiles)} unique active ingredients.")
    return df_merged

In [10]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
df = pd.read_parquet(HARMONIZED_DATA_PATH)
# get pretrained weights of MolCLR
molclr_gcn = GCN(num_layer=5, emb_dim=300, feat_dim=512, drop_ratio=0, pool="mean").to(device)
molclr_dir = Path("../MolCLR").resolve()
checkpoint_path = molclr_dir / "ckpt" / "pretrained_gcn" / "checkpoints" / "model.pth"
state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
molclr_gcn.load_state_dict(state_dict, strict=True)

encoders = {
    "GCN (128-dim)": GCNDrugEncoder(node_feature_dim=2, hidden_dim=128),
    "GIN (128-dim)": GINDrugEncoder(node_feature_dim=2, hidden_dim=128),
    "GAT (128-dim)": GATDrugEncoder(node_feature_dim=2, hidden_dim=128),
    "MolCLR (Pre-trained)": molclr_gcn
}

### Random Forest

In [13]:
results_rf = {}

# Phase-1-Baseline (Morgan FP)
results_rf["Morgan FP (Baseline)"] = early_fusion(df, target='LN_IC50', model_name='RandomForest', drug_feature_type='morgan')

# All GNN-Encoder
for name, encoder in encoders.items():
    print(f"\n================ {name} ================")
    df_gnn = add_gnn_embeddings_to_df(df, encoder, device=device)
    results_rf[name] = early_fusion(df_gnn, target='LN_IC50', model_name='RandomForest', drug_feature_type='gnn', apply_pca=True)

pd.DataFrame(results_rf).T

Starting Cross-Validation on Training Data (MORGAN)...
--- Multimodal (Random Forest - Early Fusion | MORGAN) ---
CV R² Score:   0.0753
CV RMSE:       2.6747
Test R² Score: 0.1084
Test RMSE:     2.6033
---------------------------------------------
Selected features: 990 / 996 (Chem: 18 | Bio: 972)

Top 10 most important features:
   Feature_Name  Importance                 Type
10         pca2    0.065069  Molecular structure
15         pca7    0.063487  Molecular structure
12         pca4    0.057203  Molecular structure
14         pca6    0.055488  Molecular structure
8          pca0    0.050026  Molecular structure
9          pca1    0.049675  Molecular structure
17         pca9    0.046657  Molecular structure
13         pca5    0.045956  Molecular structure
16         pca8    0.043058  Molecular structure
11         pca3    0.038287  Molecular structure

================ GCN (128-dim) ================
Embeddings created: 128 dimensions for 229 unique active ingredients.
Starting C

,cv_r2,cv_rmse,test_r2,test_rmse
Morgan FP (Baseline),0.075320,2.674681,0.108417,2.603323
GCN (128-dim),0.186354,2.509662,0.285507,2.330484
GIN (128-dim),0.194126,2.496904,0.248444,2.390165
GAT (128-dim),0.163681,2.540500,0.277763,2.343079
MolCLR (Pre-trained),0.158927,2.546501,0.270374,2.355034


### Elastic Net

In [11]:
results_en = {}

# Phase-1-Baseline (Morgan FP)
results_en["Morgan FP (Baseline)"] = early_fusion(df, target='LN_IC50', model_name='ElasticNet', drug_feature_type='morgan')

# All GNN-Encoder
for name, encoder in encoders.items():
    print(f"\n================ {name} ================")
    df_gnn = add_gnn_embeddings_to_df(df, encoder, device=device)
    results_en[name] = early_fusion(df_gnn, target='LN_IC50', model_name='ElasticNet', drug_feature_type='gnn', apply_pca=True)

pd.DataFrame(results_en).T


--- Elastic Net (Early Fusion | MORGAN) ---
Chosen Alpha:    0.594711 | L1-Ratio: 0.05
CV R² Score:     0.0637
Test R² Score:   0.1143
Test RMSE:       2.5948
---------------------------------------------
Elastic Net selected 205 / 996 features.
Top 5 Positive:
         Feature  Coefficient
10          pca2     0.298895
3     Hydrophobe     0.165036
5   PosIonizable     0.109071
2       Aromatic     0.061682
17          pca9     0.061136
Top 5 Negative:
             Feature  Coefficient
6       NegIonizable    -0.198576
15              pca7    -0.158151
4   LumpedHydrophobe    -0.108490
1           Acceptor    -0.100481
7           ZnBinder    -0.060155

================ GCN (128-dim) ================
Embeddings created: 128 dimensions for 229 unique active ingredients.

--- Elastic Net (Early Fusion | GNN) ---
Chosen Alpha:    0.393122 | L1-Ratio: 0.05
CV R² Score:     0.1236
Test R² Score:   0.2052
Test RMSE:       2.4579
---------------------------------------------
Elastic Net sel

,cv_r2,test_r2,test_rmse
Morgan FP (Baseline),0.063679,0.114252,2.594790
GCN (128-dim),0.123634,0.205248,2.457892
GIN (128-dim),0.187648,0.203726,2.460245
GAT (128-dim),0.136852,0.215094,2.442620
MolCLR (Pre-trained),0.138130,0.291664,2.320421


In [12]:
df_gnn.head()

,ModelID,SMILES,SequencingID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),...,GNN_502,GNN_503,GNN_504,GNN_505,GNN_506,GNN_507,GNN_508,GNN_509,GNN_510,GNN_511
0,ACH-000001,B([C@H](CC(C)C)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)...,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,-0.000217,0.001573,0.002739,-0.001417,0.001620,-0.000479,-0.000763,0.001795,-7.830512,0.009191
1,ACH-000001,C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0.000894,0.001484,0.001924,-0.001420,0.001359,-0.000934,0.000599,0.001659,-7.778611,0.074097
2,ACH-000001,C/C(=C\C(=O)NC1=CC=CC=C1C(=O)O)/C2=CC3=CC=CC=C...,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,-0.004215,0.001834,0.002998,0.001043,-0.000399,0.002129,-0.002797,0.003255,-4.571407,0.000849
3,ACH-000001,C/C/1=C\CC[C@]([C@H]2C[C@@H](CC[C@]3([C@@H](O3...,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0.006138,0.008100,0.005235,-0.002530,-0.004921,-0.004158,0.004032,0.006425,1.825719,0.250933
4,ACH-000001,C/C=C\1/C(=O)N[C@H](C(=O)O[C@H]\2CC(=O)N[C@@H]...,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,-0.003122,0.002173,0.001327,0.000956,0.001174,0.001953,-0.002328,0.003122,-4.843895,0.068238
